# Q5: Alternative Tokenisation Using Byte Pair Encoding (BPE)

This notebook implements an alternative tokenisation technique using **Byte Pair Encoding (BPE)** via the `tiktoken` library developed by OpenAI.  
Unlike the word-level methods in Q1 (`split()`, Regex, NLTK), BPE operates at the **subword level**, merging frequently occurring character sequences into tokens based on a large pre-trained corpus.  
The `cl100k_base` encoding is used, as it is the encoding used by GPT-4 and GPT-3.5-Turbo models (OpenAI, n.d.).

**Data source:** Data_1.txt

In [1]:
print('=' * 60)
print('CELL 1: INSTALL AND IMPORT')
print('=' * 60)

# Install tiktoken if not already installed
# tiktoken is OpenAI's open-source BPE tokeniser library
import subprocess
subprocess.run(['pip', 'install', 'tiktoken', '-q'])

import tiktoken
print('tiktoken imported successfully')
print('=' * 60)

CELL 1: INSTALL AND IMPORT
tiktoken imported successfully


In [2]:
print('=' * 60)
print('CELL 2: LOAD DATA FROM Data_1.txt')
print('=' * 60)

# Read the raw text from Data_1.txt
# The file must be in the same folder as this notebook
try:
    with open('Data_1.txt', 'r') as file:
        text = file.read()
    print('File loaded successfully')
except FileNotFoundError:
    print('Error: Data_1.txt not found. Make sure it is in the same folder as this notebook.')
    text = ''

print('\nRAW TEXT:')
print(text)
print(f'\nCharacter count        : {len(text)}')
print(f'Approximate word count : {len(text.split())}')
print('=' * 60)

CELL 2: LOAD DATA FROM Data_1.txt
File loaded successfully

RAW TEXT:
Classification is the task of choosing the correct class label for a given input. In basic
classification tasks, each input is considered in isolation from all other inputs, and the set of labels is defined in advance. The basic classification task has a number of interesting variants. For example, in multiclass classification, each instance may be assigned multiple labels; in open-class classification, the set of labels is not defined in advance; and in sequence classification, a list of inputs are jointly classified.

Character count        : 524
Approximate word count : 82


In [3]:
print('=' * 60)
print('CELL 3: BPE TOKENISATION USING TIKTOKEN')
print('=' * 60)

# Load the cl100k_base encoding
# This is the encoding used by GPT-4 and GPT-3.5-Turbo models
# It contains ~100,000 pre-trained BPE merge rules
enc = tiktoken.get_encoding('cl100k_base')

# Encode the text into BPE token IDs (integers)
tokens = enc.encode(text)

# Decode each token ID back into its readable string form
# Each decoded piece is a subword unit — not necessarily a full word
decoded = [enc.decode([token]) for token in tokens]

# Print each token with its index number
for i, token in enumerate(decoded, start=1):
    print(f'{i:5}. {token}')

print(f'\n  -> Total tokens: {len(tokens)}')
print('=' * 60)

CELL 3: BPE TOKENISATION USING TIKTOKEN
    1. Classification
    2.  is
    3.  the
    4.  task
    5.  of
    6.  choosing
    7.  the
    8.  correct
    9.  class
   10.  label
   11.  for
   12.  a
   13.  given
   14.  input
   15. .
   16.  In
   17.  basic
   18. 

   19. classification
   20.  tasks
   21. ,
   22.  each
   23.  input
   24.  is
   25.  considered
   26.  in
   27.  isolation
   28.  from
   29.  all
   30.  other
   31.  inputs
   32. ,
   33.  and
   34.  the
   35.  set
   36.  of
   37.  labels
   38.  is
   39.  defined
   40.  in
   41.  advance
   42. .
   43.  The
   44.  basic
   45.  classification
   46.  task
   47.  has
   48.  a
   49.  number
   50.  of
   51.  interesting
   52.  variants
   53. .
   54.  For
   55.  example
   56. ,
   57.  in
   58.  mult
   59. iclass
   60.  classification
   61. ,
   62.  each
   63.  instance
   64.  may
   65.  be
   66.  assigned
   67.  multiple
   68.  labels
   69. ;
   70.  in
   71.  open
   72. -

In [6]:
print('=' * 60)
print('CELL 4: KEY OBSERVATIONS')
print('=' * 60)

# Highlight specific tokens that demonstrate BPE behaviour
# These are the same benchmark cases used in Q1 for direct comparison

print('\n 1: How BPE handles "multiclass"')
# multiclass is a technical term — check if BPE keeps it whole or splits it
multiclass_tokens = [(i+1, t) for i, t in enumerate(decoded) if 'mult' in t or 'iclass' in t]
print(f'  -> {multiclass_tokens}')

print('\n 2: How BPE handles "open-class"')
# open-class is a hyphenated compound — compare with NLTK (kept whole) and Regex (hyphen lost)
openclass_tokens = [(i+1, t) for i, t in enumerate(decoded) if 'open' in t or '-class' in t]
print(f'  -> {openclass_tokens}')

print('\n 3: How BPE handles "advance" followed by punctuation')
# advance. and advance; in split() remain attached — check if BPE separates them
advance_tokens = [(i+1, t) for i, t in enumerate(decoded) if 'advance' in t or t.strip() in ['.', ';']]
print(f'  -> {advance_tokens}')

print('\n 4: Misspelled word test — "classfication" (deliberate typo)')
# BPE breaks unknown/misspelled words into subword pieces
# NLTK would treat the whole misspelled word as a single unrecognised token
typo_tokens = [enc.decode([t]) for t in enc.encode('classfication')]
print(f'  -> {typo_tokens}')
print('     Note: BPE partially recovers meaning from the misspelling')
print('     whereas NLTK would return the full misspelled string as one token.')

print('=' * 60)

CELL 4: KEY OBSERVATIONS

 1: How BPE handles "multiclass"
  -> [(58, ' mult'), (59, 'iclass'), (67, ' multiple')]

 2: How BPE handles "open-class"
  -> [(71, ' open'), (72, '-class')]

 3: How BPE handles "advance" followed by punctuation
  -> [(15, '.'), (41, ' advance'), (42, '.'), (53, '.'), (69, ';'), (83, ' advance'), (84, ';'), (97, '.')]

 4: Misspelled word test — "classfication" (deliberate typo)
  -> ['class', 'f', 'ication']
     Note: BPE partially recovers meaning from the misspelling
     whereas NLTK would return the full misspelled string as one token.
